## WEEK 2

## 1. Load Data from MySQL

In [2]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Pravith@20",
    database="retail_project"
)

query = "SELECT * FROM fact_sales"
df = pd.read_sql(query, conn)

print("Data Loaded")
df.head()

C:\Users\Pravith Kumar J\AppData\Local\Temp\ipykernel_20652\3568168547.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data Loaded


,Invoice,StockCode,CustomerID,Quantity,InvoiceDate,Price,TotalPrice
0,489434,85048,13085.0,12,2009-12-01 07:45:00,6.95,83.4
1,489434,79323P,13085.0,12,2009-12-01 07:45:00,6.75,81.0
2,489434,79323W,13085.0,12,2009-12-01 07:45:00,6.75,81.0
3,489434,22041,13085.0,48,2009-12-01 07:45:00,2.10,100.8
4,489434,21232,13085.0,24,2009-12-01 07:45:00,1.25,30.0


## 2. Data Preparation

In [3]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Reference date
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

print("Reference Date:", reference_date)

Reference Date: 2011-12-10 12:50:00


## 3. RFM Calculation

In [4]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']
rfm = rfm.reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


## 4. RFM Scoring

In [5]:
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str) +
    rfm['F_Score'].astype(str) +
    rfm['M_Score'].astype(str)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,12,77556.46,2,5,5,255
1,12347.0,2,8,5633.32,5,4,5,545
2,12348.0,75,5,2019.40,3,4,4,344
3,12349.0,19,4,4428.69,5,3,5,535
4,12350.0,310,1,334.40,2,1,2,212


## 5. Customer Segmentation

In [6]:
def segment_customer(row):
    if row['RFM_Score'] == '555':
        return 'Champions'
    elif row['R_Score'] >= 4 and row['F_Score'] >= 4:
        return 'Loyal Customers'
    elif row['R_Score'] == 5:
        return 'Recent Customers'
    elif row['F_Score'] == 5:
        return 'Frequent Customers'
    elif row['M_Score'] == 5:
        return 'Big Spenders'
    else:
        return 'At Risk'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,326,12,77556.46,2,5,5,255,Frequent Customers
1,12347.0,2,8,5633.32,5,4,5,545,Loyal Customers
2,12348.0,75,5,2019.40,3,4,4,344,At Risk
3,12349.0,19,4,4428.69,5,3,5,535,Recent Customers
4,12350.0,310,1,334.40,2,1,2,212,At Risk


## 6. Segment-Level Analysis

In [7]:
segment_summary = rfm.groupby('Segment').agg({
    'CustomerID': 'count',
    'Monetary': 'sum'
}).rename(columns={'CustomerID':'CustomerCount'})

print(segment_summary)

                    CustomerCount    Monetary
Segment                                      
At Risk                      3632  2498390.05
Big Spenders                  147   848456.96
Champions                     474  8356881.90
Frequent Customers            268  1632652.17
Loyal Customers              1008  3936256.75
Recent Customers              352   470791.33


## 7. Cohort Analysis (Retention Analysis)

In [8]:
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

cohort = df.groupby('CustomerID')['InvoiceDate'].min().reset_index()
cohort['CohortMonth'] = cohort['InvoiceDate'].dt.to_period('M')

df = df.merge(cohort[['CustomerID','CohortMonth']], on='CustomerID')

df['CohortIndex'] = (df['InvoiceMonth'] - df['CohortMonth']).apply(lambda x: x.n)

cohort_data = df.groupby(['CohortMonth','CohortIndex'])['CustomerID'].nunique().reset_index()

cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

print(cohort_pivot)

CohortIndex     0      1      2      3      4      5      6      7      8   \
CohortMonth                                                                  
2009-12      955.0  337.0  319.0  406.0  363.0  343.0  360.0  327.0  321.0   
2010-01      383.0   79.0  119.0  117.0  101.0  115.0   99.0   88.0  107.0   
2010-02      376.0   89.0   84.0  109.0   92.0   75.0   72.0  107.0   95.0   
2010-03      443.0   84.0  102.0  107.0  103.0   90.0  109.0  134.0  122.0   
2010-04      294.0   57.0   57.0   48.0   54.0   66.0   81.0   77.0   31.0   
2010-05      254.0   40.0   43.0   44.0   45.0   65.0   54.0   32.0   15.0   
2010-06      270.0   47.0   51.0   55.0   62.0   77.0   34.0   24.0   22.0   
2010-07      186.0   29.0   34.0   55.0   54.0   26.0   21.0   27.0   27.0   
2010-08      162.0   33.0   48.0   52.0   28.0   19.0   16.0   20.0   22.0   
2010-09      243.0   55.0   57.0   30.0   22.0   25.0   33.0   24.0   31.0   
2010-10      377.0   97.0   55.0   47.0   33.0   31.0   49.0   5

## 8. Market Basket Analysis

In [9]:
basket = df.groupby(['Invoice','StockCode'])['Quantity'].sum().unstack().fillna(0)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

print(basket.head())

C:\Users\Pravith Kumar J\AppData\Local\Temp\ipykernel_20652\3096200925.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


StockCode  10002  10080  10109  10120  10123C  10123G  10124A  10124G  10125  \
Invoice                                                                        
489434         0      0      0      0       0       0       0       0      0   
489435         0      0      0      0       0       0       0       0      0   
489436         0      0      0      0       0       0       0       0      0   
489437         1      0      0      0       0       0       0       0      0   
489438         0      0      0      0       0       0       0       0      0   

StockCode  10133  ...  BANK CHARGES  C2  D  DOT  M  PADS  POST  SP1002  \
Invoice           ...                                                    
489434         0  ...             0   0  0    0  0     0     0       0   
489435         0  ...             0   0  0    0  0     0     0       0   
489436         0  ...             0   0  0    0  0     0     0       0   
489437         0  ...             0   0  0    0  0     0     0       

## 9. Save RFM Data to MySQL

In [10]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customer_rfm (
    CustomerID FLOAT,
    Recency INT,
    Frequency INT,
    Monetary DECIMAL(10,2),
    R_Score INT,
    F_Score INT,
    M_Score INT,
    RFM_Score VARCHAR(10),
    Segment VARCHAR(50)
)
""")

cursor.execute("TRUNCATE TABLE customer_rfm")

data = rfm.values.tolist()

query = """
INSERT INTO customer_rfm 
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

cursor.executemany(query, data)
conn.commit()

print("RFM Data Saved to MySQL")

RFM Data Saved to MySQL


## 10. Final Output Check

In [11]:
cursor.execute("SELECT COUNT(*) FROM customer_rfm")
print("Total Customers:", cursor.fetchone()[0])

print("\nSegment Distribution:")
print(rfm['Segment'].value_counts())

Total Customers: 5881

Segment Distribution:
Segment
At Risk               3632
Loyal Customers       1008
Champions              474
Recent Customers       352
Frequent Customers     268
Big Spenders           147
Name: count, dtype: int64
